# Model Context Protocol

## Environment Variables & Imports

Before using this notebook, please ensure you have obtained an API Key from your LLM backend and update the `.env` file to include it as follows:

```bash
GOOGLE_API_KEY=<copy API key here>
OPENAI_API_KEY=<copy API key here>
ANTRHOPIC_API_KEY=<copy API key here>
LLAMA_CPP_API_KEY=<copy API key here>

In [ ]:
import initialize_notebook # noqa

In [ ]:
import json
import pathlib

import jinja2

from hslu.dlm03.common import backend as backend_lib
from hslu.dlm03.common import chat as chat_lib
from hslu.dlm03.common import presets
from hslu.dlm03.common.displays import ipython_display
from hslu.dlm03.tools import lint
from hslu.dlm03.util import file as file_lib

## Parameters

In [ ]:
backend_name = "Gemma 4 26B"
backend_config = presets.CONFIGS_BY_NAME[backend_name]
backend = backend_lib.LLM.from_config(backend_config)

In [ ]:
mcp_url = "http://localhost:8000/mcp"

# Agentic Harness

In [ ]:
import re
def remove_thoughts(text: str) -> str:
    # Remove well-formed <thought>...</thought> blocks (multiline safe)
    cleaned = re.sub(r"<thought>.*?</thought>", "", text, flags=re.DOTALL | re.IGNORECASE)

    # Optionally handle unclosed <thought> tags (strip until end)
    cleaned = re.sub(r"<thought>.*$", "", cleaned, flags=re.DOTALL | re.IGNORECASE)

    return cleaned.strip()

In [ ]:
import mcp
from mcp.client import streamable_http

from hslu.dlm03.common import tools as tools_lib


async with tools_lib.mcp_session(mcp_url) as session:
    await session.initialize()
    mcp_tools = await session.list_tools()
    tools = [tools_lib.tool_from_mcp(tool) for tool in mcp_tools.tools]

In [ ]:
system_instructions_template_string = \
"""You are an epxert DevOps engineer, specialized in the Plan phase.
Your purpose is to help transform bug reports into features and tasks for the team to work on.
You will be provided with some vague requirements, and your goal is to fill the Project Management software (OpenProject) with new tasks in order to solve this request.
Please be thorough in your assessment of what is required to be done, and fill out the work time estimation,
and figure out who should be assigned to this task based on current work assignments.
Please keep the backlog organized (with the proper overall milestone, summary tasks and tasks), and do it retroactively if needed."""
system_instructions_template = jinja2.Template(system_instructions_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_requests = [
    "Having a web UI to check your balance would be great!" ,
    "My account does not allow me to go below 0 CHF despite having this option in my contract",
    "It would be nice to be able to group accounts together to have an overall view of our current assets",
    "I would love to be able to see the payments I have made over time and how my account balance evlolved",
]

In [ ]:
user_message_template_string = \
"""User request: {{ user_request }}"""
user_message_template = jinja2.Template(user_message_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
system_instructions = system_instructions_template.render()
for user_request in user_requests:
    chat_display = ipython_display.IPythonChatDisplay()
    user_message = user_message_template.render(user_request=user_request)
    chat_display.show()
    chat = chat_lib.Chat(
        messages = [
            {"role": "system", "content": system_instructions},
            {"role": "user", "content": user_message},
        ],
        observers=[chat_display],
    )
    async with tools_lib.mcp_session(mcp_url) as session:
        await session.initialize()
        mcp_tools = await session.list_tools()
        tools = [tools_lib.tool_from_mcp(tool) for tool in mcp_tools.tools]
        done = False
        while not done:
            response = backend.generate(
                chat,
                tools=tools,
            )
            done = True
            message = response.choices[0].message
            if message.content is not None:
                message.content = remove_thoughts(message.content)
            chat.append(message)
            for tool_call in message.tool_calls or ():
                done = False
                tool_name = tool_call.function.name
                arguments = json.loads(tool_call.function.arguments)
                tool_call_result = await session.call_tool(tool_name, arguments)
                for content in tool_call_result.content:
                    tool_call_result_content = tools_lib.tool_call_result_from_mcp(
                        tool_call.id,
                        content,
                    )
                    chat.append(tool_call_result_content)